# SARIMAX Forecasting Playground

This notebook provides an interactive environment to test the cluster-based SARIMAX (Seasonal AutoRegressive Integrated Moving Average with eXogenous factors) model.

### Key Features
- **Dual-Mode Pipeline**: Support for both `long_term` (weather-only) and `day_ahead` (lag-enriched) forecasting.
- **Seasonality Analysis**: ACF/PACF plots and decomposition to validate model parameters.
- **Modular Architecture**: Uses decoupled functions from `src/models/sarimax_model.py`.
- **Performance Optimization**: Aggregates data by cluster to ensure efficient training.

## 0. Environment Setup

Resolve project-level modules from the `src` directory and load the necessary forecasting functions.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import specialized SARIMAX functions
from src.models.sarimax_model import (
    load_processed_data, 
    preprocess_and_split, 
    train_models, 
    predict_models, 
    evaluate_models,
    save_sarimax_artifacts
)
from src.tools.visualization import plot_cluster_portfolio, analyze_time_periods

print("✅ Setup complete. Local SARIMAX modules loaded from src/.")

## 1. Global Data Loading

We load the global processed dataset once.

In [ ]:
dataset_path = os.path.join(PROJECT_ROOT, 'Datasets', 'processed_electricity_data.parquet')
df_long = load_processed_data(dataset_path)

print(f"\n Total observations loaded: {len(df_long):,}")

---

## 2. PART 1: Seasonality & Autocorrelation Analysis

To justify our Choice of SARIMAX parameters $(p, d, q) \times (P, D, Q, s)$, we analyze the average profile of Cluster 0. We expect strong autocorrelation at 24h intervals (96 intervals of 15min).

In [ ]:
# Aggregating average profile for Cluster 0
cluster_0_profile = (
    df_long[df_long['Cluster'] == 0]
    .groupby('Date')['Consumption'].mean()
    .sort_index()
    .asfreq('15min')
    .fillna(method='ffill')
)

# 1. Seasonal Decomposition (looking at a 1-week window for clarity)
result = seasonal_decompose(cluster_0_profile[:96*7], model='additive', period=96)
fig = result.plot()
fig.set_size_inches(14, 8)
plt.title("Seasonal Decomposition - Cluster 0", fontsize=16)
plt.show()

In [ ]:
# 2. ACF and PACF Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(cluster_0_profile, lags=96*2, ax=axes[0])
axes[0].set_title("Autocorrelation (ACF) - 2 Days Lag")

plot_pacf(cluster_0_profile, lags=96*2, ax=axes[1])
axes[1].set_title("Partial Autocorrelation (PACF) - 2 Days Lag")

plt.show()

---

## 3. PART 2: EXPERIMENT A - Long-Term Horizon (Weather Only)

This mode benchmarks the model's ability to predict load based strictly on seasonal trends and weather signals, without the benefit of recent consumption history.

In [ ]:
# A.1 Prepare Data (Long Term)
train_agg_lt, test_agg_lt, test_raw_lt, scalers_lt, sw_lt, regs_lt = preprocess_and_split(df_long, mode='long_term')

# A.2 Train Cluster Models (takes a few minutes)
models_lt = train_models(train_agg_lt, regs_lt)

# A.3 Predict & Un-scale
results_lt = predict_models(models_lt, test_agg_lt, test_raw_lt, scalers_lt, regs_lt)

# A.4 Evaluate
eval_lt, summary_lt = evaluate_models(results_lt)
display(summary_lt)

# Save Artifacts
save_sarimax_artifacts(models_lt, scalers_lt, sw_lt, regs_lt, mode='long_term')

In [ ]:
# Visualization for Long-Term results
plot_cluster_portfolio(eval_lt, summary_lt, model_label="SARIMAX (Long-Term)")

---

## 4. PART 3: EXPERIMENT B - Day-Ahead Horizon (Weather + Lags)

In this mode, we provide SARIMAX with historical consumption lags (24h and 1-week back) as exogenous variables. This should significantly improve performance on short-term horizons.

In [ ]:
# B.1 Prepare Data (Day Ahead)
train_agg_da, test_agg_da, test_raw_da, scalers_da, sw_da, regs_da = preprocess_and_split(df_long, mode='day_ahead')

# B.2 Train Cluster Models
models_da = train_models(train_agg_da, regs_da)

# B.3 Predict & Un-scale
results_da = predict_models(models_da, test_agg_da, test_raw_da, scalers_da, regs_da)

# B.4 Evaluate
eval_da, summary_da = evaluate_models(results_da)
display(summary_da)

# Save Artifacts
save_sarimax_artifacts(models_da, scalers_da, sw_da, regs_da, mode='day_ahead')

In [ ]:
# Visualization for Day-Ahead results
plot_cluster_portfolio(eval_da, summary_da, model_label="SARIMAX (Day-Ahead)")

---

## 5. Summary Comparison

Comparison of Portfolio WMAPE for both experiments.

In [ ]:
comparison_df = pd.DataFrame({
    "Long-Term (Weather Only)": summary_lt["Portfolio_WMAPE"],
    "Day-Ahead (Weather + Lags)": summary_da["Portfolio_WMAPE"]
}).style.background_gradient(cmap='RdYlGn_r', axis=1)

print("Portfolio WMAPE Comparison by Cluster:")
display(comparison_df)